In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
声子寿命专项分析（含完整数据导出）
包含：
- 寿命分布图（线性/对数） + 对应直方图CSV
- 寿命 vs 频率（散点）
- 寿命 vs 频率（分段统计） + 分段统计CSV
- 寿命 vs 平均自由程（散点）
- 多温度寿命对比（分段均值曲线） + 多温度分段统计CSV
- 专门的CSV导出（完整/绘图/按能带/总体/多温度散点）
"""
import h5py
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

# 创建输出目录
output_dir = 'phonon_lifetime_analysis'
os.makedirs(output_dir, exist_ok=True)

print("="*80)
print("声子寿命专项分析")
print("="*80)

# 读取数据
with h5py.File('kappa-m111111.hdf5', 'r') as f:
    temperature = f['temperature'][:]
    frequency = f['frequency'][:]
    gamma = f['gamma'][:]
    group_velocity = f['group_velocity'][:]
    mode_kappa = f['mode_kappa'][:]
    qpoint = f['qpoint'][:]
    weight = f['weight'][:]

# ============================================================================
# 第一部分：单温度详细分析
# ============================================================================

temp_idx = 6  # 例如 300K（请根据你的数据确认该索引对应的温度）
print(f"\n分析温度: {temperature[temp_idx]:.0f}K")

nqpoints, nbands = frequency.shape

# 计算基本量
gamma_at_T = gamma[temp_idx]                           # THz
scattering_rate = 2.0 * gamma_at_T                     # THz

# 【修正】计算声子寿命（数值单位=ps）
# 推导：1 THz = 1/ps，因此 τ[ps] = 1 / (2π * Γ[THz])
lifetime = np.zeros_like(scattering_rate)
mask_valid = scattering_rate > 0
lifetime[mask_valid] = 1.0 / (2.0 * np.pi * scattering_rate[mask_valid])  # ps

# 群速度与平均自由程（nm）
gv_norm = np.linalg.norm(group_velocity, axis=2)       # m/s
# mfp[nm] = v[m/s] * τ[ps] * (1e-12 s/ps) * (1e9 nm/m) = v * τ * 1e-3
mean_free_path = gv_norm * lifetime * 1e-3             # nm

# 热导率（各方向平均）
kappa_avg = (mode_kappa[temp_idx, :, :, 0] +
             mode_kappa[temp_idx, :, :, 1] +
             mode_kappa[temp_idx, :, :, 2]) / 3.0

# 准备导出数据（逐模态平铺）
qpoint_indices = np.repeat(np.arange(nqpoints), nbands)
band_indices = np.tile(np.arange(nbands), nqpoints)

df_lifetime = pd.DataFrame({
    'qpoint_idx': qpoint_indices,
    'band_idx': band_indices,
    'qx': np.repeat(qpoint[:, 0], nbands),
    'qy': np.repeat(qpoint[:, 1], nbands),
    'qz': np.repeat(qpoint[:, 2], nbands),
    'frequency_THz': frequency.flatten(),
    'gamma_THz': gamma_at_T.flatten(),
    'scattering_rate_THz': scattering_rate.flatten(),
    'lifetime_ps': lifetime.flatten(),
    'group_velocity_m_per_s': gv_norm.flatten(),
    'mean_free_path_nm': mean_free_path.flatten(),
    'kappa_avg_W_per_mK': kappa_avg.flatten()
})

# 导出完整数据
csv_path = os.path.join(output_dir, f'lifetime_complete_{temperature[temp_idx]:.0f}K.csv')
df_lifetime.to_csv(csv_path, index=False)
print(f"✓ 完整寿命数据: {csv_path}")

# 过滤有效数据（用于绘图/统计）
mask_plot = (df_lifetime['frequency_THz'] > 0.1) & (df_lifetime['lifetime_ps'] > 0)
df_lifetime_plot = df_lifetime[mask_plot].copy()

csv_plot_path = os.path.join(output_dir, f'lifetime_plot_{temperature[temp_idx]:.0f}K.csv')
df_lifetime_plot.to_csv(csv_plot_path, index=False)
print(f"✓ 绘图用寿命数据: {csv_plot_path}")

# ============================================================================
# 第二部分：声子寿命可视化（含数据导出）
# ============================================================================

print("\n生成声子寿命图表...")

# 图1: 寿命 vs 频率（主图；颜色=群速度，log 归一化可能要求群速度>0）
fig, ax = plt.subplots(figsize=(12, 7))
sc = ax.scatter(df_lifetime_plot['frequency_THz'],
                df_lifetime_plot['lifetime_ps'],
                s=10, alpha=0.6,
                c=df_lifetime_plot['group_velocity_m_per_s'],
                cmap='plasma',
                norm=plt.matplotlib.colors.LogNorm())
ax.set_xlabel('Frequency (THz)', fontsize=14)
ax.set_ylabel('Phonon Lifetime (ps)', fontsize=14)
ax.set_title(f'Phonon Lifetime vs Frequency at {temperature[temp_idx]:.0f}K\n'
             f'(colored by group velocity)', fontsize=16)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
cbar = plt.colorbar(sc, ax=ax, label='Group Velocity (m/s)')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, f'lifetime_vs_freq_{temperature[temp_idx]:.0f}K.png'), dpi=300)
plt.close()
print("  ✓ 寿命vs频率图")

# 图2: 寿命分布直方图（线性 & log10 尺度）
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 线性尺度
ax1 = axes[0]
ax1.hist(df_lifetime_plot['lifetime_ps'], bins=50, alpha=0.7,
         edgecolor='black', color='steelblue')
ax1.set_xlabel('Lifetime (ps)', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title('Lifetime Distribution (Linear Scale)', fontsize=13)
ax1.axvline(df_lifetime_plot['lifetime_ps'].mean(),
            color='red', linestyle='--', linewidth=2, label='Mean')
ax1.axvline(df_lifetime_plot['lifetime_ps'].median(),
            color='orange', linestyle='--', linewidth=2, label='Median')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 对数尺度（以 log10(lifetime) 作直方图）
ax2 = axes[1]
ax2.hist(np.log10(df_lifetime_plot['lifetime_ps']), bins=50, alpha=0.7,
         edgecolor='black', color='coral')
ax2.set_xlabel('log₁₀(Lifetime) [ps]', fontsize=12)
ax2.set_ylabel('Count', fontsize=12)
ax2.set_title('Lifetime Distribution (Log Scale)', fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, f'lifetime_distribution_{temperature[temp_idx]:.0f}K.png'), dpi=300)
plt.close()
print("  ✓ 寿命分布直方图")

# === 图2 对应：导出直方图数据（线性 & log10）===
# 使用 numpy.histogram / plt.hist 的等价计数与边界信息
counts_lin, edges_lin = np.histogram(df_lifetime_plot['lifetime_ps'].values, bins=50)
hist_lin_df = pd.DataFrame({
    'bin_left_ps': edges_lin[:-1],
    'bin_right_ps': edges_lin[1:],
    'bin_center_ps': (edges_lin[:-1] + edges_lin[1:]) / 2,
    'count': counts_lin.astype(int)
})
csv_hist_lin = os.path.join(output_dir, f'lifetime_hist_linear_{temperature[temp_idx]:.0f}K.csv')
hist_lin_df.to_csv(csv_hist_lin, index=False)

log_vals = np.log10(df_lifetime_plot['lifetime_ps'].values)
counts_log, edges_log = np.histogram(log_vals, bins=50)
hist_log_df = pd.DataFrame({
    'bin_left_log10_ps': edges_log[:-1],
    'bin_right_log10_ps': edges_log[1:],
    'bin_center_log10_ps': (edges_log[:-1] + edges_log[1:]) / 2,
    'count': counts_log.astype(int)
})
csv_hist_log = os.path.join(output_dir, f'lifetime_hist_log10_{temperature[temp_idx]:.0f}K.csv')
hist_log_df.to_csv(csv_hist_log, index=False)
print("  ✓ 直方图计数CSV：",
      os.path.basename(csv_hist_lin), ",", os.path.basename(csv_hist_log))

# 图3: 寿命 vs 频率（分段统计）
freq_bins = np.linspace(df_lifetime_plot['frequency_THz'].min(),
                        df_lifetime_plot['frequency_THz'].max(), 20)
bin_indices = np.digitize(df_lifetime_plot['frequency_THz'], freq_bins)

freq_centers = []
lifetime_means = []
lifetime_stds = []
lifetime_medians = []
bin_lefts = []
bin_rights = []
counts_per_bin = []

for i in range(1, len(freq_bins)):
    mask_bin = bin_indices == i
    if mask_bin.sum() > 0:
        left_i = freq_bins[i-1]
        right_i = freq_bins[i]
        vals = df_lifetime_plot.loc[mask_bin, 'lifetime_ps'].values

        bin_lefts.append(left_i)
        bin_rights.append(right_i)
        freq_centers.append((left_i + right_i) / 2)
        lifetime_means.append(vals.mean())
        lifetime_stds.append(vals.std())
        lifetime_medians.append(np.median(vals))
        counts_per_bin.append(int(mask_bin.sum()))

fig, ax = plt.subplots(figsize=(12, 7))
ax.errorbar(freq_centers, lifetime_means, yerr=lifetime_stds,
            fmt='o-', linewidth=2, markersize=6, capsize=5,
            label='Mean ± Std', color='steelblue')
ax.plot(freq_centers, lifetime_medians, 's--', linewidth=2, markersize=6,
        label='Median', color='coral')
ax.set_xlabel('Frequency (THz)', fontsize=14)
ax.set_ylabel('Phonon Lifetime (ps)', fontsize=14)
ax.set_title(f'Binned Average Lifetime vs Frequency at {temperature[temp_idx]:.0f}K', fontsize=16)
ax.set_yscale('log')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, f'lifetime_binned_{temperature[temp_idx]:.0f}K.png'), dpi=300)
plt.close()
print("  ✓ 分段平均寿命图")

# === 图3 对应：导出分段统计CSV（含边界/中心/样本数/均值/中位数/标准差）===
binned_df = pd.DataFrame({
    'bin_left_THz': bin_lefts,
    'bin_right_THz': bin_rights,
    'freq_center_THz': freq_centers,
    'count': counts_per_bin,
    'lifetime_mean_ps': lifetime_means,
    'lifetime_std_ps': lifetime_stds,
    'lifetime_median_ps': lifetime_medians
})
csv_binned = os.path.join(output_dir, f'lifetime_binned_stats_{temperature[temp_idx]:.0f}K.csv')
binned_df.to_csv(csv_binned, index=False)
print("  ✓ 分段统计CSV：", os.path.basename(csv_binned))

# 图4: 寿命 vs 平均自由程
fig, ax = plt.subplots(figsize=(10, 7))
sc = ax.scatter(df_lifetime_plot['lifetime_ps'],
                df_lifetime_plot['mean_free_path_nm'],
                s=10, alpha=0.6,
                c=df_lifetime_plot['frequency_THz'],
                cmap='viridis')
ax.set_xlabel('Lifetime (ps)', fontsize=14)
ax.set_ylabel('Mean Free Path (nm)', fontsize=14)
ax.set_title(f'Lifetime vs Mean Free Path at {temperature[temp_idx]:.0f}K', fontsize=16)
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.colorbar(sc, ax=ax, label='Frequency (THz)')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, f'lifetime_vs_mfp_{temperature[temp_idx]:.0f}K.png'), dpi=300)
plt.close()
print("  ✓ 寿命vs平均自由程图")

# ============================================================================
# 第三部分：多温度寿命对比（含分段统计导出）
# ============================================================================

print("\n生成多温度对比...")

# 选择几个代表性温度（请按你的数据范围调整）
temp_indices = [2, 6, 10, 14, 18]  # 例如 100K, 300K, 500K, 700K, 900K

all_temp_data = []          # 多温度：散点级数据
all_temp_binned_data = []   # 多温度：分段统计数据（用于导出）

fig, ax = plt.subplots(figsize=(12, 8))

for idx in temp_indices:
    gamma_T = gamma[idx]                     # THz
    scatter_T = 2.0 * gamma_T                # THz
    lifetime_T = np.zeros_like(scatter_T)
    mask_T = scatter_T > 0
    # 寿命（ps）
    lifetime_T[mask_T] = 1.0 / (2.0 * np.pi * scatter_T[mask_T])

    freq_flat = frequency.flatten()
    life_flat = lifetime_T.flatten()
    mask_valid = (freq_flat > 0.1) & (life_flat > 0)

    # 保存散点级数据（用于外部拟合/分析）
    temp_df = pd.DataFrame({
        'temperature_K': temperature[idx],
        'frequency_THz': freq_flat[mask_valid],
        'lifetime_ps': life_flat[mask_valid]
    })
    all_temp_data.append(temp_df)

    # 与绘图一致的分段统计（均值/标准差/样本数）
    freq_bins_T = np.linspace(0, freq_flat[mask_valid].max(), 30)
    bin_idx_T = np.digitize(freq_flat[mask_valid], freq_bins_T)

    bin_lefts_T, bin_rights_T, freq_c, life_m, life_s, counts_T = [], [], [], [], [], []
    for i in range(1, len(freq_bins_T)):
        m = bin_idx_T == i
        if m.sum() > 0:
            left_i, right_i = freq_bins_T[i-1], freq_bins_T[i]
            vals = life_flat[mask_valid][m]
            bin_lefts_T.append(left_i)
            bin_rights_T.append(right_i)
            freq_c.append((left_i + right_i) / 2)
            life_m.append(vals.mean())
            life_s.append(vals.std())
            counts_T.append(int(m.sum()))

    # 绘制每一条温度曲线（使用分段均值）
    ax.plot(freq_c, life_m, 'o-', linewidth=2, markersize=4,
            label=f'{temperature[idx]:.0f}K', alpha=0.8)

    # 记录到跨温度分段统计表
    all_temp_binned_data.append(pd.DataFrame({
        'temperature_K': temperature[idx],
        'bin_left_THz': bin_lefts_T,
        'bin_right_THz': bin_rights_T,
        'freq_center_THz': freq_c,
        'count': counts_T,
        'lifetime_mean_ps': life_m,
        'lifetime_std_ps': life_s
    }))

ax.set_xlabel('Frequency (THz)', fontsize=14)
ax.set_ylabel('Mean Lifetime (ps)', fontsize=14)
ax.set_title('Temperature Dependence of Phonon Lifetime', fontsize=16)
ax.set_yscale('log')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'lifetime_vs_temperature.png'), dpi=300)
plt.close()
print("  ✓ 温度依赖图")

# 导出多温度：散点级数据
df_multi_temp = pd.concat(all_temp_data, ignore_index=True)
csv_multi = os.path.join(output_dir, 'lifetime_multi_temperature.csv')
df_multi_temp.to_csv(csv_multi, index=False)
print(f"✓ 多温度寿命数据(散点): {csv_multi}")

# 导出多温度：分段统计数据
df_multi_binned = pd.concat(all_temp_binned_data, ignore_index=True)
csv_multi_binned = os.path.join(output_dir, 'lifetime_binned_by_temperature.csv')
df_multi_binned.to_csv(csv_multi_binned, index=False)
print(f"✓ 多温度分段统计CSV: {csv_multi_binned}")

# ============================================================================
# 第四部分：统计摘要
# ============================================================================

print("\n生成统计摘要...")

# 按能带分组统计（使用 df_lifetime_plot 中的有效样本）
band_stats = []
for ib in range(nbands):
    mask_band = df_lifetime_plot['band_idx'] == ib
    if mask_band.sum() > 0:
        band_stats.append({
            'band_index': ib,
            'count': int(mask_band.sum()),
            'lifetime_mean_ps': df_lifetime_plot.loc[mask_band, 'lifetime_ps'].mean(),
            'lifetime_std_ps': df_lifetime_plot.loc[mask_band, 'lifetime_ps'].std(),
            'lifetime_min_ps': df_lifetime_plot.loc[mask_band, 'lifetime_ps'].min(),
            'lifetime_max_ps': df_lifetime_plot.loc[mask_band, 'lifetime_ps'].max(),
            'lifetime_median_ps': df_lifetime_plot.loc[mask_band, 'lifetime_ps'].median(),
            'freq_mean_THz': df_lifetime_plot.loc[mask_band, 'frequency_THz'].mean(),
            'gv_mean_m_per_s': df_lifetime_plot.loc[mask_band, 'group_velocity_m_per_s'].mean()
        })

df_band_stats = pd.DataFrame(band_stats)
csv_stats = os.path.join(output_dir, f'lifetime_by_band_{temperature[temp_idx]:.0f}K.csv')
df_band_stats.to_csv(csv_stats, index=False)
print(f"✓ 按能带统计: {csv_stats}")

# 总体统计
overall_stats = {
    'temperature_K': [float(temperature[temp_idx])],
    'total_modes': [int(len(df_lifetime_plot))],
    'lifetime_mean_ps': [df_lifetime_plot['lifetime_ps'].mean()],
    'lifetime_std_ps': [df_lifetime_plot['lifetime_ps'].std()],
    'lifetime_min_ps': [df_lifetime_plot['lifetime_ps'].min()],
    'lifetime_max_ps': [df_lifetime_plot['lifetime_ps'].max()],
    'lifetime_median_ps': [df_lifetime_plot['lifetime_ps'].median()],
    'lifetime_q25_ps': [df_lifetime_plot['lifetime_ps'].quantile(0.25)],
    'lifetime_q75_ps': [df_lifetime_plot['lifetime_ps'].quantile(0.75)],
    'mfp_mean_nm': [df_lifetime_plot['mean_free_path_nm'].mean()],
    'mfp_median_nm': [df_lifetime_plot['mean_free_path_nm'].median()]
}
df_overall = pd.DataFrame(overall_stats)
csv_overall = os.path.join(output_dir, f'lifetime_summary_{temperature[temp_idx]:.0f}K.csv')
df_overall.to_csv(csv_overall, index=False)
print(f"✓ 总体统计: {csv_overall}")

# ============================================================================
# 最终总结
# ============================================================================

print("\n" + "="*80)
print("声子寿命分析完成！")
print("="*80)
print(f"\n输出目录: {output_dir}/")
print(f"\n生成的文件:")
print("  CSV文件:")
print(f"    - lifetime_complete_{temperature[temp_idx]:.0f}K.csv (完整数据)")
print(f"    - lifetime_plot_{temperature[temp_idx]:.0f}K.csv (绘图数据)")
print(f"    - lifetime_hist_linear_{temperature[temp_idx]:.0f}K.csv (直方图-线性)")
print(f"    - lifetime_hist_log10_{temperature[temp_idx]:.0f}K.csv (直方图-log10)")
print(f"    - lifetime_binned_stats_{temperature[temp_idx]:.0f}K.csv (单温度分段统计)")
print(f"    - lifetime_multi_temperature.csv (多温度散点)")
print(f"    - lifetime_binned_by_temperature.csv (多温度分段统计)")
print(f"    - lifetime_by_band_{temperature[temp_idx]:.0f}K.csv (按能带统计)")
print(f"    - lifetime_summary_{temperature[temp_idx]:.0f}K.csv (总体统计)")
print("\n  图片文件:")
print(f"    - lifetime_vs_freq_{temperature[temp_idx]:.0f}K.png")
print(f"    - lifetime_distribution_{temperature[temp_idx]:.0f}K.png")
print(f"    - lifetime_binned_{temperature[temp_idx]:.0f}K.png")
print(f"    - lifetime_vs_mfp_{temperature[temp_idx]:.0f}K.png")
print("    - lifetime_vs_temperature.png")
print(f"\n关键统计 (at {temperature[temp_idx]:.0f}K):")
print(f"  平均寿命: {df_lifetime_plot['lifetime_ps'].mean():.3e} ps")
print(f"  中位数寿命: {df_lifetime_plot['lifetime_ps'].median():.3e} ps")
print(f"  寿命范围: {df_lifetime_plot['lifetime_ps'].min():.3e} - "
      f"{df_lifetime_plot['lifetime_ps'].max():.3e} ps")
print(f"  平均自由程: {df_lifetime_plot['mean_free_path_nm'].mean():.2f} nm")
print("="*80)


声子寿命专项分析

分析温度: 300K
✓ 完整寿命数据: phonon_lifetime_analysis\lifetime_complete_300K.csv
✓ 绘图用寿命数据: phonon_lifetime_analysis\lifetime_plot_300K.csv

生成声子寿命图表...
  ✓ 寿命vs频率图
  ✓ 寿命分布直方图
  ✓ 直方图计数CSV： lifetime_hist_linear_300K.csv , lifetime_hist_log10_300K.csv
  ✓ 分段平均寿命图
  ✓ 分段统计CSV： lifetime_binned_stats_300K.csv
  ✓ 寿命vs平均自由程图

生成多温度对比...
  ✓ 温度依赖图
✓ 多温度寿命数据(散点): phonon_lifetime_analysis\lifetime_multi_temperature.csv
✓ 多温度分段统计CSV: phonon_lifetime_analysis\lifetime_binned_by_temperature.csv

生成统计摘要...
✓ 按能带统计: phonon_lifetime_analysis\lifetime_by_band_300K.csv
✓ 总体统计: phonon_lifetime_analysis\lifetime_summary_300K.csv

声子寿命分析完成！

输出目录: phonon_lifetime_analysis/

生成的文件:
  CSV文件:
    - lifetime_complete_300K.csv (完整数据)
    - lifetime_plot_300K.csv (绘图数据)
    - lifetime_hist_linear_300K.csv (直方图-线性)
    - lifetime_hist_log10_300K.csv (直方图-log10)
    - lifetime_binned_stats_300K.csv (单温度分段统计)
    - lifetime_multi_temperature.csv (多温度散点)
    - lifetime_binned_by_temperature.csv (多温度分段统计)
    - 